In [2]:
!find /kaggle/input -name "edufineweb_*.npy"

/kaggle/input/notebooks/hrishikeshdhuri19/prep-data-gpt300m/edu_fineweb300M/edufineweb_train_000002.npy
/kaggle/input/notebooks/hrishikeshdhuri19/prep-data-gpt300m/edu_fineweb300M/edufineweb_val_000000.npy
/kaggle/input/notebooks/hrishikeshdhuri19/prep-data-gpt300m/edu_fineweb300M/edufineweb_train_000003.npy
/kaggle/input/notebooks/hrishikeshdhuri19/prep-data-gpt300m/edu_fineweb300M/edufineweb_train_000001.npy


In [3]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_REPO"] = "Hrishikesh1903/gpt124m-ckpt"
print(os.environ["HF_REPO"])

Hrishikesh1903/gpt124m-ckpt


In [6]:
%%writefile train_gpt2_300m.py
"""
~300M-parameter GPT-2 style model, trained from scratch on FineWeb-Edu shards.
Built for Kaggle 2x T4 (fp16 autocast + GradScaler, DDP, torch.compile).

Launch:
    torchrun --standalone --nproc_per_node=2 train_gpt2_300m.py

Env vars (all optional):
    MODEL_SIZE    "300m" (304M params, default) or "124m" (GPT-2 small)
    DATA_ROOT     folder with edufineweb_*.npy shards   (default edu_fineweb300M)
    TOTAL_TOKENS  token budget for the LR schedule      (default 3e8)
    MICRO_BATCH   per-GPU micro-batch                   (default 4)
    MAX_HOURS     stop gracefully + checkpoint after N hours (default 0 = off)
    CKPT_DIR      where checkpoints/logs go             (default log)
    RESUME_FROM   checkpoint to resume from             (default CKPT_DIR/ckpt_last.pt if it exists)
    CKPT_EVERY_MIN  save (and upload) a checkpoint every N minutes (default 30)
    HF_REPO       e.g. "user/gpt300m-ckpt": private Hugging Face repo used as persistent storage
    HF_TOKEN      Hugging Face token with WRITE access (use Kaggle Secrets)

With HF_REPO set: on start, if no local checkpoint exists, the latest one is downloaded from the
repo and training resumes from it; during training every checkpoint is uploaded in the background.
"""
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import math
import time
import inspect
import datetime
import threading
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.distributed as dist
from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import tiktoken


# ----------------------------------------------------------------------------
# model
# ----------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304   # 50257 padded to a multiple of 128
    n_layer: int = 20
    n_head: int = 16
    n_embd: int = 1024        # -> ~304M parameters


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"sequence length {T} > block size {self.config.block_size}"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, device_type, verbose=True):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for p in param_dict.values() if p.dim() >= 2]
        nodecay_params = [p for p in param_dict.values() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == "cuda"
        if verbose:
            print(f"decayed tensors: {len(decay_params)} ({sum(p.numel() for p in decay_params):,} params), "
                  f"non-decayed: {len(nodecay_params)} ({sum(p.numel() for p in nodecay_params):,} params), "
                  f"fused AdamW: {use_fused}")
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)


# ----------------------------------------------------------------------------
# data: sharded, rank-aware loader
# ----------------------------------------------------------------------------
def load_tokens(filename):
    npt = np.load(filename).astype(np.int32)
    return torch.tensor(npt, dtype=torch.long)


class DataLoaderLite:
    def __init__(self, B, T, process_rank, num_processes, split, data_root, verbose=True):
        self.B, self.T = B, T
        self.process_rank, self.num_processes = process_rank, num_processes
        assert split in {'train', 'val'}
        shards = sorted(s for s in os.listdir(data_root) if split in s and s.endswith('.npy'))
        assert len(shards) > 0, f"no {split} shards found in {data_root}"
        self.shards = [os.path.join(data_root, s) for s in shards]
        if verbose:
            print(f"found {len(self.shards)} shards for split {split}")
        self.reset()

    def reset(self):
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position: self.current_position + B * T + 1]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.current_position += B * T * self.num_processes
        if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_position = B * T * self.process_rank
        return x, y

    # for resuming: all ranks advance in lockstep, so (shard, rank-0 position) is enough
    def state(self):
        return {'shard': self.current_shard,
                'base': self.current_position - self.B * self.T * self.process_rank}

    def load_state(self, s):
        self.current_shard = s['shard']
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = s['base'] + self.B * self.T * self.process_rank


# ----------------------------------------------------------------------------
# DDP setup
# ----------------------------------------------------------------------------
ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    assert torch.cuda.is_available(), "DDP needs CUDA"
    init_process_group(backend='nccl', timeout=datetime.timedelta(minutes=60))
    ddp_rank = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])
    device = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)
else:
    ddp_rank, ddp_local_rank, ddp_world_size = 0, 0, 1
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
master_process = ddp_rank == 0
device_type = 'cuda' if device.startswith('cuda') else 'cpu'

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

enc = tiktoken.get_encoding('gpt2')

# ----------------------------------------------------------------------------
# hyperparameters
# ----------------------------------------------------------------------------
data_root = os.environ.get("DATA_ROOT", "edu_fineweb300M")
total_tokens = int(float(os.environ.get("TOTAL_TOKENS", 3e8)))
total_batch_size = 262144                       # 2**18 tokens per optimizer step
MODEL_PRESETS = {
    "124m": dict(cfg=dict(n_layer=12, n_head=12, n_embd=768),  micro_batch=16, max_lr=6e-4),
    "300m": dict(cfg=dict(n_layer=20, n_head=16, n_embd=1024), micro_batch=4,  max_lr=3e-4),
}
model_size = os.environ.get("MODEL_SIZE", "300m").lower()
assert model_size in MODEL_PRESETS, f"MODEL_SIZE must be one of {list(MODEL_PRESETS)}"
preset = MODEL_PRESETS[model_size]
B = int(os.environ.get("MICRO_BATCH", preset["micro_batch"]))   # per-GPU micro-batch
T = 1024
max_hours = float(os.environ.get("MAX_HOURS", 0))
use_compile = True

assert total_batch_size % (B * T * ddp_world_size) == 0, "total_batch_size must be divisible by B*T*world_size"
grad_accum_steps = total_batch_size // (B * T * ddp_world_size)

max_steps = total_tokens // total_batch_size
warmup_steps = max(20, int(0.04 * max_steps))
max_lr = preset["max_lr"]                        # GPT-3 settings: 6e-4 (124M), 3e-4 (~350M)
min_lr = max_lr * 0.1
weight_decay = 0.1

val_every = 250
val_B = min(B, 4)                    # eval runs uncompiled: full fp16 logits + fp32 loss are memory hungry
val_steps = 20 * B // val_B          # keeps the same number of validation tokens as before
sample_every = 500
ckpt_every_min = float(os.environ.get("CKPT_EVERY_MIN", 30))

log_dir = os.environ.get("CKPT_DIR", "log")
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "log.txt")
ckpt_path = os.path.join(log_dir, "ckpt_last.pt")
resume_path = os.environ.get("RESUME_FROM", ckpt_path)

if master_process:
    print(f"model {model_size} | world size {ddp_world_size} | micro-batch {B}x{T} | grad accum {grad_accum_steps} | "
          f"tokens/step {total_batch_size:,} | max_steps {max_steps} | warmup {warmup_steps}")

def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

# ----------------------------------------------------------------------------
# build everything
# ----------------------------------------------------------------------------
train_loader = DataLoaderLite(B, T, ddp_rank, ddp_world_size, 'train', data_root, verbose=master_process)
val_loader = DataLoaderLite(val_B, T, ddp_rank, ddp_world_size, 'val', data_root, verbose=master_process)

model = GPT(GPTConfig(**preset["cfg"])).to(device)
raw_model = model        # plain, uncompiled GPT: used for eval, sampling, checkpoints
if master_process:
    print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

optimizer = raw_model.configure_optimizers(weight_decay, max_lr, device_type, verbose=master_process)
scaler = torch.amp.GradScaler('cuda')

# ----------------------------------------------------------------------------
# optional remote persistence (Hugging Face Hub): survives Kaggle session loss
# ----------------------------------------------------------------------------
hf_repo = os.environ.get("HF_REPO", "")
hf_token = os.environ.get("HF_TOKEN", "")
_upload_thread = None


def hf_pull():
    """master only: download ckpt_last.pt (+ log.txt) from the repo if it exists."""
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=hf_token)
    if not api.file_exists(hf_repo, "ckpt_last.pt"):
        print(f"no checkpoint in {hf_repo} yet: starting fresh")
        return
    # any error here propagates on purpose: starting fresh by mistake would overwrite the remote checkpoint
    hf_hub_download(hf_repo, "ckpt_last.pt", token=hf_token, local_dir=log_dir)
    print(f"downloaded ckpt_last.pt from {hf_repo}")
    if api.file_exists(hf_repo, "log.txt"):
        hf_hub_download(hf_repo, "log.txt", token=hf_token, local_dir=log_dir)


def hf_push(final=False, include_final_model=False):
    """master only: upload the latest checkpoint in a background thread (each upload is an atomic commit)."""
    global _upload_thread
    if _upload_thread is not None and _upload_thread.is_alive():
        if not final:
            print("[upload] previous upload still running, skipping this one")
            return
        _upload_thread.join()
    snap = ckpt_path + ".upload"
    if os.path.exists(snap):
        os.remove(snap)
    os.link(ckpt_path, snap)      # stable inode: safe even if ckpt_last.pt is replaced mid-upload
    files = [(snap, "ckpt_last.pt"), (log_file, "log.txt")]
    if include_final_model:
        files.append((os.path.join(log_dir, "model_final.pt"), "model_final.pt"))

    def work():
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        try:
            for local, remote in files:
                api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=hf_repo)
            print(f"[upload] pushed {', '.join(r for _, r in files)} to {hf_repo}")
        except Exception as e:
            print(f"[upload failed] {e}")
            return
        finally:
            if os.path.exists(snap):
                os.remove(snap)
        try:
            api.super_squash_history(repo_id=hf_repo)   # drop older checkpoint versions to save storage
        except Exception as e:
            print(f"[squash skipped] {e}")

    _upload_thread = threading.Thread(target=work)
    _upload_thread.start()
    if final:
        _upload_thread.join()


if hf_repo:
    assert hf_token, "HF_REPO is set but HF_TOKEN is empty"
    if master_process:
        from huggingface_hub import HfApi
        HfApi(token=hf_token).create_repo(hf_repo, private=True, exist_ok=True)   # fails fast on a bad/read-only token
        if not os.path.exists(resume_path):
            hf_pull()
    if ddp:
        dist.barrier()

start_step = 0
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device, weights_only=True)
    assert ckpt['config'] == asdict(raw_model.config), (
        f"checkpoint config {ckpt['config']} does not match MODEL_SIZE={model_size}; "
        f"use a different CKPT_DIR / HF_REPO for a different model")
    raw_model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
    train_loader.load_state(ckpt['loader'])
    start_step = ckpt['step']
    del ckpt
    if master_process:
        print(f"resumed from {resume_path} at step {start_step}")
elif master_process:
    with open(log_file, "w"):
        pass

if use_compile:
    model = torch.compile(model)
if ddp:
    model = DDP(model, device_ids=[ddp_local_rank])


def save_checkpoint(next_step, val_loss=None, final=False):
    ckpt = {
        'model': raw_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'config': asdict(raw_model.config),
        'step': next_step,
        'loader': train_loader.state(),
        'val_loss': val_loss,
    }
    tmp = ckpt_path + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, ckpt_path)                      # atomic: never leaves a half-written file
    if final:
        torch.save({'model': raw_model.state_dict(), 'config': asdict(raw_model.config)},
                   os.path.join(log_dir, "model_final.pt"))


sample_rng = torch.Generator(device=device)
sample_rng.manual_seed(42)

@torch.no_grad()
def generate(prompt, max_new_tokens=60, num_seqs=3, top_k=50, temperature=0.8):
    raw_model.eval()
    tokens = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device)
    tokens = tokens.unsqueeze(0).repeat(num_seqs, 1)
    for _ in range(max_new_tokens):
        with torch.autocast(device_type=device_type, dtype=torch.float16):
            logits, _ = raw_model(tokens[:, -raw_model.config.block_size:])
        logits = logits[:, -1, :].float() / temperature
        logits[:, 50257:] = float('-inf')            # padded vocab ids are not real tokens
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1, generator=sample_rng)
        tokens = torch.cat((tokens, torch.gather(topk_idx, -1, ix)), dim=1)
    raw_model.train()
    return [enc.decode(row.tolist()) for row in tokens]


# ----------------------------------------------------------------------------
# train
# ----------------------------------------------------------------------------
t_start = time.time()
last_ckpt_time = time.time()
val_loss_value = None

for step in range(start_step, max_steps):
    t0 = time.time()
    last_step = (step == max_steps - 1)

    # --- validation loss ---
    if step % val_every == 0 or last_step:
        if device_type == 'cuda':
            torch.cuda.empty_cache()
        raw_model.eval()
        val_loader.reset()
        with torch.no_grad():
            val_loss_accum = torch.zeros(1, device=device)
            for _ in range(val_steps):
                x, y = val_loader.next_batch()
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type=device_type, dtype=torch.float16):
                    _, loss = raw_model(x, y)
                val_loss_accum += loss.detach() / val_steps
        raw_model.train()
        if device_type == 'cuda':
            torch.cuda.empty_cache()
        if ddp:
            dist.all_reduce(val_loss_accum, op=dist.ReduceOp.AVG)
        val_loss_value = val_loss_accum.item()
        if master_process:
            print(f"validation loss: {val_loss_value:.4f}")
            with open(log_file, "a") as f:
                f.write(f"{step} val {val_loss_value:.4f}\n")

    # --- sample text ---
    if master_process and ((step % sample_every == 0 and step > 0) or last_step):
        for i, s in enumerate(generate("The most important thing to understand about")):
            print(f"> sample {i}: {s!r}")

    # --- one optimizer step (gradient accumulation) ---
    optimizer.zero_grad(set_to_none=True)
    loss_accum = torch.zeros(1, device=device)
    for micro_step in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)
        if ddp:   # all-reduce gradients only on the last micro-step
            model.require_backward_grad_sync = (micro_step == grad_accum_steps - 1)
        with torch.autocast(device_type=device_type, dtype=torch.float16):
            _, loss = model(x, y)
        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        scaler.scale(loss).backward()
    if ddp:
        dist.all_reduce(loss_accum, op=dist.ReduceOp.AVG)
    scaler.unscale_(optimizer)                       # unscale BEFORE clipping
    norm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    scaler.step(optimizer)
    scaler.update()
    if device_type == 'cuda':
        torch.cuda.synchronize()
    dt = time.time() - t0
    tok_per_sec = total_batch_size / dt

    if master_process:
        msg = (f"step {step:5d}/{max_steps} | loss {loss_accum.item():.6f} | lr {lr:.4e} | "
               f"norm {norm:.4f} | scale {scaler.get_scale():.0f} | dt {dt*1000:.0f}ms | tok/s {tok_per_sec:.0f}")
        if step == start_step + 1 and device_type == 'cuda':
            msg += f" | peak mem {torch.cuda.max_memory_allocated() / 1e9:.1f}GB"
        print(msg)
        with open(log_file, "a") as f:
            f.write(f"{step} train {loss_accum.item():.6f}\n")

    # --- time limit (so a Kaggle session ends cleanly with a checkpoint) ---
    stop = torch.zeros(1, device=device)
    if max_hours > 0 and (time.time() - t_start) > max_hours * 3600:
        stop += 1
    if ddp:
        dist.all_reduce(stop, op=dist.ReduceOp.MAX)
    stop = stop.item() > 0

    # --- checkpoint ---
    ckpt_due = (time.time() - last_ckpt_time) > ckpt_every_min * 60
    if master_process and (ckpt_due or last_step or stop):
        save_checkpoint(step + 1, val_loss_value, final=last_step)
        last_ckpt_time = time.time()
        print(f"saved checkpoint at step {step + 1}")
        if hf_repo:
            hf_push(final=(last_step or stop), include_final_model=last_step)
    if stop:
        if master_process:
            print(f"MAX_HOURS reached: stopping at step {step + 1}. Re-run to resume.")
        break

if ddp:
    destroy_process_group()

Overwriting train_gpt2_300m.py


In [7]:
!MODEL_SIZE=124m MAX_HOURS=10.5 DATA_ROOT=/kaggle/input/notebooks/hrishikeshdhuri19/prep-data-gpt300m/edu_fineweb300M CKPT_DIR=/kaggle/working/log124 torchrun --standalone --nproc_per_node=2 train_gpt2_300m.py

W1011 17:05:29.883000 449 torch/distributed/run.py:851] 
W1011 17:05:29.883000 449 torch/distributed/run.py:851] *****************************************
W1011 17:05:29.883000 449 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1011 17:05:29.883000 449 torch/distributed/run.py:851] *****************************************
model 124m | world size 2 | micro-batch 16x1024 | grad accum 8 | tokens/step 262,144 | max_steps 1144 | warmup 45
found 3 shards for split train
found 1 shards for split val
parameters: 124.5M
decayed tensors: 50 (124,354,560 params), non-decayed: 98 (121,344 params), fused AdamW: True
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` t

In [9]:
%%writefile generate.py

"""
Generate text from the trained GPT-2 style model (model_final.pt).

Kaggle / Colab usage:
    import os
    os.environ["HF_TOKEN"] = "hf_..."          # or UserSecretsClient().get_secret("HF_TOKEN")
    !python generate.py --prompt "The most important thing to understand about"

Local file instead of Hugging Face:
    !python generate.py --ckpt /kaggle/working/log/model_final.pt --prompt "..."
"""
import os
import argparse
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken


# ---------------------------------------------------------------- model (same as training)
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        return self.lm_head(self.transformer.ln_f(x))


# ---------------------------------------------------------------- loading
def load_model(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model = GPT(GPTConfig(**ckpt["config"]))
    model.load_state_dict(ckpt["model"])
    return model.to(device).eval()


def get_ckpt(args):
    if args.ckpt:
        return args.ckpt
    from huggingface_hub import hf_hub_download
    return hf_hub_download(args.repo, args.filename, token=os.environ.get("HF_TOKEN"))


# ---------------------------------------------------------------- sampling
@torch.no_grad()
def generate(model, enc, prompt, device, max_new_tokens=150, num_seqs=3, temperature=0.8,
             top_k=50, top_p=0.95, repetition_penalty=1.1, seed=None):
    gen = torch.Generator(device=device)
    if seed is not None:
        gen.manual_seed(seed)
    else:
        gen.seed()

    tokens = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device)
    tokens = tokens.unsqueeze(0).repeat(num_seqs, 1)
    finished = torch.zeros(num_seqs, dtype=torch.bool, device=device)
    eot = enc.eot_token

    for _ in range(max_new_tokens):
        ctx = tokens[:, -model.config.block_size:]
        with torch.autocast(device_type=device.split(":")[0], dtype=torch.float16,
                            enabled=device.startswith("cuda")):
            logits = model(ctx)
        logits = logits[:, -1, :].float()
        logits[:, 50257:] = float("-inf")                 # padded vocab ids are not real tokens

        if repetition_penalty != 1.0:                      # discourage already-used tokens
            score = torch.gather(logits, 1, tokens)
            score = torch.where(score < 0, score * repetition_penalty, score / repetition_penalty)
            logits.scatter_(1, tokens, score)

        if temperature <= 0:                               # greedy
            next_tok = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                kth = torch.topk(logits, top_k, dim=-1).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            if top_p < 1.0:                                # nucleus sampling
                sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
                cum = sorted_p.cumsum(dim=-1)
                sorted_p[(cum - sorted_p) > top_p] = 0.0
                sorted_p /= sorted_p.sum(dim=-1, keepdim=True)
                pick = torch.multinomial(sorted_p, 1, generator=gen)
                next_tok = torch.gather(sorted_i, -1, pick)
            else:
                next_tok = torch.multinomial(probs, 1, generator=gen)

        next_tok = torch.where(finished[:, None], torch.full_like(next_tok, eot), next_tok)
        tokens = torch.cat((tokens, next_tok), dim=1)
        finished |= (next_tok.squeeze(1) == eot)
        if finished.all():
            break

    outs = []
    for row in tokens.tolist():
        if eot in row[len(enc.encode(prompt)):]:
            row = row[:len(enc.encode(prompt)) + row[len(enc.encode(prompt)):].index(eot)]
        outs.append(enc.decode(row))
    return outs


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--prompt", default="The most important thing to understand about")
    p.add_argument("--ckpt", default="", help="local model_final.pt; if empty, download from HF")
    p.add_argument("--repo", default="Hrishikesh1903/gpt124m-ckpt")
    p.add_argument("--filename", default="model_final.pt")
    p.add_argument("--max_new_tokens", type=int, default=150)
    p.add_argument("--num_seqs", type=int, default=3)
    p.add_argument("--temperature", type=float, default=0.8)
    p.add_argument("--top_k", type=int, default=50)
    p.add_argument("--top_p", type=float, default=0.95)
    p.add_argument("--repetition_penalty", type=float, default=1.1)
    p.add_argument("--seed", type=int, default=None)
    args = p.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    enc = tiktoken.get_encoding("gpt2")
    model = load_model(get_ckpt(args), device)
    print(f"loaded model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params on {device}\n")

    outs = generate(model, enc, args.prompt, device, args.max_new_tokens, args.num_seqs,
                    args.temperature, args.top_k, args.top_p, args.repetition_penalty, args.seed)
    for i, o in enumerate(outs):
        print(f"--- sample {i} ---\n{o}\n")

Writing generate.py


In [10]:
!python generate.py --prompt "The most important thing to understand about"


model_final.pt: downloading bytes: ██▋                      | 54.5MB, 4.18MB/s  
model_final.pt: downloading bytes: ███████████▎             |  226MB, 14.5MB/s  
model_final.pt: downloading bytes: ████████████▏            |  244MB, 15.0MB/s  
model_final.pt: downloading bytes: ████████████▍            |  249MB, 15.2MB/s  
model_final.pt: downloading bytes: ████████████▉            |  258MB, 15.5MB/s  
model_final.pt: downloading bytes: ████████████████▌        |  329MB, 18.2MB/s  
model_final.pt: downloading bytes: ████████████████▊        |  334MB, 18.5MB/s  
model_final.pt: downloading bytes: █████████████████████    |  420MB, 22.5MB/s  
model_final.pt: downloading bytes: ██████████████████████▎  |  443MB, 23.1MB/s  
model_final.pt: reconstructing file:  81%|███████▎ |  402MB /  498MB, 23.6MB/s  
model_final.pt: downloading bytes: ██████████████████████▍  |  447MB, 23.1MB/s  
model_final.pt: downloading bytes: ███████████████████████▎ |  465MB, 23.4MB/s  
model_final.pt: downloading

In [14]:
import os, torch, tiktoken
from huggingface_hub import hf_hub_download
from generate import load_model, generate

# token: environment variable if set, otherwise Kaggle Secrets
token = os.environ.get("HF_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")

path = hf_hub_download("Hrishikesh1903/gpt124m-ckpt", "model_final.pt", token=token)

device = "cuda" if torch.cuda.is_available() else "cpu"
enc = tiktoken.get_encoding("gpt2")
model = load_model(path, device)
print("model loaded on", device)

model loaded on cuda


In [16]:
prompts = [
    "Photosynthesis is the process by which plants",
    "The French Revolution began in 1789 when",
    "In this lesson, students will learn how to",
    "The human heart is divided into four chambers:",
    "One of the main causes of climate change is",
    "To calculate the area of a triangle, you need to",
    "The Industrial Revolution had a major impact on",
    "Water is essential for life because",
    "The Pythagorean theorem states that",
    "A healthy diet should include a variety of",
]

for p in prompts:
    print(">>>", p)
    print(generate(model, enc, p, device, max_new_tokens=100, num_seqs=1,
                   temperature=0.7, top_k=40, top_p=0.9, repetition_penalty=1.15)[0])
    print("-" * 60)

>>> Photosynthesis is the process by which plants
Photosynthesis is the process by which plants are produced in a variety of different parts. The plant has several different components:
- To build and maintain a balanced diet, it can be grown in several types:
- To grow up to 8 inches tall, to produce 10 pounds per day, and then, in the same manner as a result, it will have a great amount of nutrients and nutrients and also provide for many health benefits.
- If you don’t get enough vitamins and minerals from your diet, there are some
------------------------------------------------------------
>>> The French Revolution began in 1789 when
The French Revolution began in 1789 when the United States was elected. This had been the first to be the first state to become the first state of England.
The British Empire was formed by the King Henry VIII and was known as the “Indian Revolution”. The British Empire began to revolutionize the country with the help of a free, strong army. This resul